In [ ]:
import numpy as np
import mne
from moabb.datasets import Lee2019_ERP
from moabb.paradigms import P300
import gc
import pandas as pd
from EEGInception import EEGInception

# dataset
dataset=Lee2019_ERP()

# setup of preprocessing parameters for the P300 paradigm:
# - Sampling rate a 100Hz (resample=100)
# - Bandpass filter 0.5-45Hz (fmin=0.5, fmax=45.0)
# - Time window cut to 1 second (tmin=0.0, tmax=1.0)
paradigm = P300(
    resample=100,
    fmin=0.5,
    fmax=45.0,
    tmin=0.0,
    tmax=1.0,
    baseline=None 
)

# define the list of subjects to load
lista_soggetti = [1,2,3,4,5] 

# list of channels to drop (non-standard, non-EEG, or noisy channels)

channels_to_drop = ['TP9', 'TP10', 'PO9', 'PO10', 'FT9', 'FTT9h', 'TTP7h', 'TPP9h', 'FT10', 'FTT10h', 'TPP8h', 'TPP10h', 'F9', 'F10']

X_list = []
y_list = []
meta_list = []

print("Start incremental loading ...")

for sbj in lista_soggetti:
    print(f"--- Processing of subject {sbj} ---")
    
    # single extraction for the current subject (returns already preprocessed data)
    X_tmp, y_tmp, meta_tmp = paradigm.get_data(dataset=dataset, subjects=[sbj])
    
    # retrieve channel names (only the first time or for each subject if they differ)
    if sbj == lista_soggetti[0]:
        subj_data = dataset.get_data(subjects=[sbj])[sbj]
        session_name = list(subj_data.keys())[0]
        run_name = list(subj_data[session_name].keys())[0]
        ch_names = subj_data[session_name][run_name].ch_names
        drop_indices = [ch_names.index(ch) for ch in channels_to_drop if ch in ch_names]

    # 2. remove unwanted channels by index (if they exist in the current subject)
    X_tmp = np.delete(X_tmp, drop_indices, axis=1)
    
    # 3. convert to float32 to save memory
    X_tmp = X_tmp.astype('float32')
    
    X_list.append(X_tmp)
    y_list.append(y_tmp)
    meta_list.append(meta_tmp)
    
    # Memory cleanup after each subject
    del X_tmp
    gc.collect()

# Final joining of the blocks
X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\n Loading complete")
print(f"New filtered shape (X): {X.shape} -> (tests, channels, samples)")
print(f"shape of labels (y): {y.shape}")

# 4. EEGInception model initialization
n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))


model = EEGInception(
    in_shape=(n_channels, n_samples),
    n_out=n_classes,
    alignment='latent', # alignment strategy
    dropout=0.25
)

print("\nEEGInception model initialized successfully")

Inizio caricamento incrementale (Soggetto per Soggetto)...
--- Elaborazione Soggetto 1 ---
--- Elaborazione Soggetto 2 ---
--- Elaborazione Soggetto 3 ---
--- Elaborazione Soggetto 4 ---
--- Elaborazione Soggetto 5 ---

✅ Caricamento completato!
Nuova shape filtrata (X): (41400, 48, 100) -> (prove, canali, campioni)
Shape etichette (y): (41400,)

Modello EEGInception inizializzato con successo!


New filtered shape (X): (41400, 48, 100) -> (trials, channels, samples)

Label shape (y): (41400)

In [ ]:
import torch
from torch.utils.data import TensorDataset

# 1. identification of labels (MOABB uses string labels(Target and NonTarget), we need to convert them to integers for PyTorch)
labels_dict = {val: i for i, val in enumerate(np.unique(y))}
y_int = np.array([labels_dict[label] for label in y], dtype=np.int64)

# Conversion to PyTorch tensors
X_tensor = torch.from_numpy(X)  # (tests, channels, samples)
y_tensor = torch.from_numpy(y_int)  # (tests)

# Creation of the base Dataset
dataset_pytorch = TensorDataset(X_tensor, y_tensor)

print(f"X tensor ready: {X_tensor.shape} | Type: {X_tensor.dtype}")
print(f"y tensor ready: {y_tensor.shape} | Type: {y_tensor.dtype}")

# 4. Quick forward pass test with a small batch of data
# We extract the first 12 trials for testing
X_batch_test = X_tensor[:12]
sbj_trials_test = 12

# The model returns the unnormalized logits for each class
output_test = model(X_batch_test, sbj_trials_test)
print(f"\nShape of the model output: {output_test.shape} -> (tests, classes)")

Tensore X pronto: torch.Size([41400, 48, 100]) | Tipo: torch.float32
Tensore y pronto: torch.Size([41400]) | Tipo: torch.int64

Shape dell'output del modello: torch.Size([12, 2]) -> (prove, classi)


```text
X Tensor ready: torch.Size([41400, 48, 100]) | Type: torch.float32
y Tensor ready: torch.Size([41400]) | Type: torch.int64
Model output shape: torch.Size([12, 2]) -> (trials, classes)

In [ ]:
import random
import torch.nn as nn
from torch.utils.data import Sampler, DataLoader

class P300BalancedBatchSampler(Sampler):
    """
    Customized sampler to create batches with:
    - 4 subjects per batch
    - 12 trials per subject (10 Non-Target, 2 Target)
    """
    def __init__(self, metadata, y_int, n_subjects_per_batch=4, n_trials_per_subject=12):
        self.metadata = metadata.reset_index(drop=True)
        self.y = y_int
        self.n_subjects_per_batch = n_subjects_per_batch
        
        # Each subject has 12 trials, with a 5:1 ratio -> 10 Non-Target (class 0), 2 Target (class 1)
        self.n_non_target = 10
        self.n_target = 2
        
        # We group indices by subject and class
        self.indices_by_subj = {}
        self.subjects = self.metadata['subject'].unique()
        
        for subj in self.subjects:
            # find the global indices where the subject is the current one
            subj_mask = (self.metadata['subject'] == subj).values
            subj_indices = np.where(subj_mask)[0]
            
            # For this subject, we separate the indices of Non-Target (0) and Target (1)
            y_subj = self.y[subj_indices]
            self.indices_by_subj[subj] = {
                0: subj_indices[y_subj == 0].tolist(),
                1: subj_indices[y_subj == 1].tolist()
            }
            
    def __iter__(self):
        available_subjects = list(self.subjects)
        
        # create a shuffled copy of the indices to iterate over without always picking the same ones
        iter_indices = {}
        for subj in available_subjects:
            iter_indices[subj] = {
                0: random.sample(self.indices_by_subj[subj][0], len(self.indices_by_subj[subj][0])),
                1: random.sample(self.indices_by_subj[subj][1], len(self.indices_by_subj[subj][1]))
            }
            
        # Calculating how many complete batches we can generate (limited by the Target class, which is rare)
        min_targets = min([len(self.indices_by_subj[subj][1]) for subj in available_subjects])
        n_batches = (min_targets // self.n_target) * (len(available_subjects) // self.n_subjects_per_batch)
        
        for _ in range(n_batches):
            batch_indices = []
            
            # We sample 4 subjects for this batch (with replacement if we have few loaded)
            if len(available_subjects) >= self.n_subjects_per_batch:
                batch_subjs = random.sample(available_subjects, self.n_subjects_per_batch)
            else:
                batch_subjs = random.choices(available_subjects, k=self.n_subjects_per_batch)
            
            for subj in batch_subjs:
                # If we have exhausted the indices for this subject, we reshuffle and "reload" them
                if len(iter_indices[subj][0]) < self.n_non_target:
                    iter_indices[subj][0] = random.sample(self.indices_by_subj[subj][0], len(self.indices_by_subj[subj][0]))
                if len(iter_indices[subj][1]) < self.n_target:
                    iter_indices[subj][1] = random.sample(self.indices_by_subj[subj][1], len(self.indices_by_subj[subj][1]))
                
                # Extract exactly 10 Non-Target and 2 Target
                selected_non_targets = iter_indices[subj][0][:self.n_non_target]
                iter_indices[subj][0] = iter_indices[subj][0][self.n_non_target:] # remove them
                
                selected_targets = iter_indices[subj][1][:self.n_target]
                iter_indices[subj][1] = iter_indices[subj][1][self.n_target:] # remove them
                
                batch_indices.extend(selected_non_targets + selected_targets)
                
            yield batch_indices

    def __len__(self):
        min_targets = min([len(self.indices_by_subj[subj][1]) for subj in self.subjects])
        return (min_targets // self.n_target) * (len(self.subjects) // self.n_subjects_per_batch)


# --- 5. Initialize the Dataloader and the Loss Function ---

# create the Sampler by passing the metadata (which contains the subject info)
custom_sampler = P300BalancedBatchSampler(metadata, y_int, n_subjects_per_batch=4)

# create the DataLoader for Training
train_loader = DataLoader(dataset_pytorch, batch_sampler=custom_sampler)

# test the DataLoader by fetching the first batch
for X_batch, y_batch in train_loader:
    print(f"\nShape X_batch from DataLoader: {X_batch.shape} -> (4 subjects * 12 tests, channels, samples)")
    print(f"Class distribution in the batch (1=Target, 0=Non-Target):")
    print(f"Target: {(y_batch == 1).sum().item()} | Non-Target: {(y_batch == 0).sum().item()}")
    break # stop after the first batch for testing

# calculation of weights for the Loss Function (Inverse of the distribution in the training dataset)
n_0 = (y_int == 0).sum()
n_1 = (y_int == 1).sum()
# weights are proportional to the inverse of the frequency to counteract the imbalance
weight_0 = 1.0 / n_0
weight_1 = 1.0 / n_1
weights = torch.tensor([weight_0, weight_1], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
print(f"\nWeights for Loss calculated: Non-Target={weight_0:.6f}, Target={weight_1:.6f}")


Shape X_batch dal DataLoader: torch.Size([48, 48, 100]) -> (4 soggetti * 12 prove, canali, campioni)
Distribuzione classi nel batch (1=Target, 0=Non-Target):
Target: 8 | Non-Target: 40

Pesi per la Loss calcolati: Non-Target=0.000029, Target=0.000145


```text
X_batch shape from DataLoader: torch.Size([48, 48, 100]) -> (4 subjects * 12 trials, channels, samples)
Class distribution in batch (1=Target, 0=Non-Target): Target: 8 | Non-Target: 40
Calculated Loss weights: Non-Target=0.000029, Target=0.000145

In [ ]:
import torch
import os

# Check how many cores are available and set them
num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
print(f"PyTorch configured to use {num_cores} threads.")

PyTorch configurato per usare 6 threads.


In [ ]:
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score
import time

# --- LOSO TRAINING SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Parameters from the paper
num_epochs = 10 
learning_rate = 0.001

# Variables to store results
loso_results = []

# Get the unique list of loaded subjects (e.g., [1, 2, 3, 4, 5])
subjects_array = metadata['subject'].unique()

print("\n=== Starting LOSO (Leave-One-Subject-Out) Validation ===")

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    # 1. DATA SPLIT: Training (N-1) and Test (1)
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    X_train = X_tensor[train_mask]
    y_train = y_tensor[train_mask]
    meta_train = metadata[train_mask]
    
    X_test = X_tensor[test_mask]
    y_test = y_tensor[test_mask]
    meta_test = metadata[test_mask]
    
    # Create Datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    # Create Sampler for Training only
    train_sampler = P300BalancedBatchSampler(meta_train, y_train.numpy(), n_subjects_per_batch=4)
    train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=3)
    
    # For Testing, we can use a standard DataLoader or evaluate the entire subject at once
    # We choose a batch size of 12 for consistency with the model and to calculate test statistics
    test_loader = DataLoader(test_dataset, batch_size=12, shuffle=False, num_workers=3)
    
    # 2. MODEL INITIALIZATION (New model for each test subject)
    model = EEGInception(
        in_shape=(n_channels, n_samples),
        n_out=n_classes,
        alignment='latent', 
        dropout=0.25
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 3. TRAINING LOOP
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad(set_to_none=True)  # Optimization for PyTorch 2.0+
            
            # Note on sbj_trials: In the unbalanced train_loader it is always 12
            outputs = model(batch_X, sbj_trials=12)
            
            loss = criterion(outputs, batch_y.to(device))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")
            
    # 4. EVALUATION ON TEST SUBJECT
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            # If the last batch has fewer than 12 trials, we must adjust sbj_trials
            current_batch_size = batch_X.shape[0]
            
            batch_X = batch_X.to(device)
            
            outputs = model(batch_X, sbj_trials=current_batch_size)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(batch_y.numpy())
            
    # 5. METRIC CALCULATION
    bal_acc = balanced_accuracy_score(all_targets, all_preds)
    loso_results.append(bal_acc)
    
    print(f"-> Finished Test Subject {test_subject} | Balanced Acc: {bal_acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")
    
# --- FINAL RESULTS ---
print("\n" + "="*40)
print("FINAL LOSO RESULTS (Latent Alignment)")
print(f"Mean Balanced Accuracy: {np.mean(loso_results)*100:.2f}% ± {np.std(loso_results)*100:.2f}%")
print("="*40)

Utilizzando il device: cpu

=== Inizio Validazione LOSO (Leave-One-Subject-Out) ===

-> Addestramento per Test Subject: 1
   Epoca [5/10], Loss: 0.2084
   Epoca [10/10], Loss: 0.1595
-> Fine Test Subject 1 | Balanced Acc: 76.49% | Tempo: 1593.7s

-> Addestramento per Test Subject: 2
   Epoca [5/10], Loss: 0.2915
   Epoca [10/10], Loss: 0.2452
-> Fine Test Subject 2 | Balanced Acc: 95.41% | Tempo: 1607.4s

-> Addestramento per Test Subject: 3
   Epoca [5/10], Loss: 0.2766
   Epoca [10/10], Loss: 0.2257
-> Fine Test Subject 3 | Balanced Acc: 90.38% | Tempo: 1604.1s

-> Addestramento per Test Subject: 4
   Epoca [5/10], Loss: 0.2408
   Epoca [10/10], Loss: 0.2021
-> Fine Test Subject 4 | Balanced Acc: 70.54% | Tempo: 1606.2s

-> Addestramento per Test Subject: 5
   Epoca [5/10], Loss: 0.2131
   Epoca [10/10], Loss: 0.1723
-> Fine Test Subject 5 | Balanced Acc: 72.61% | Tempo: 1542.3s

RISULTATI FINALI LOSO (Latent Alignment)
Accuratezza Bilanciata Media: 81.09% ± 9.96%


```text
Using device: cpu

=== Starting LOSO (Leave-One-Subject-Out) Validation ===

-> Training for Test Subject: 1
   Epoch [5/10], Loss: 0.2084
   Epoch [10/10], Loss: 0.1595
-> Finished Test Subject 1 | Balanced Acc: 76.49% | Time: 1593.7s

-> Training for Test Subject: 2
   Epoch [5/10], Loss: 0.2915
   Epoch [10/10], Loss: 0.2452
-> Finished Test Subject 2 | Balanced Acc: 95.41% | Time: 1607.4s

-> Training for Test Subject: 3
   Epoch [5/10], Loss: 0.2766
   Epoch [10/10], Loss: 0.2257
-> Finished Test Subject 3 | Balanced Acc: 90.38% | Time: 1604.1s

-> Training for Test Subject: 4
   Epoch [5/10], Loss: 0.2408
   Epoch [10/10], Loss: 0.2021
-> Finished Test Subject 4 | Balanced Acc: 70.54% | Time: 1606.2s

-> Training for Test Subject: 5
   Epoch [5/10], Loss: 0.2131
   Epoch [10/10], Loss: 0.1723
-> Finished Test Subject 5 | Balanced Acc: 72.61% | Time: 1542.3s

============================================================
FINAL LOSO RESULTS (Latent Alignment)
Mean Balanced Accuracy: 81.09% ± 9.96%
============================================================

In [ ]:
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score
import time

# --- LOSO TRAINING SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Parameters taken from the paper
num_epochs = 10 
learning_rate = 0.001

# Variables to store results
loso_results = []

# Get the unique list of loaded subjects (e.g., [1, 2, 3, 4, 5])
subjects_array = metadata['subject'].unique()

print("\n=== Starting LOSO (Leave-One-Subject-Out) Validation ===")

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    # 1. DATA SPLIT: Training (N-1) and Test (1)
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    X_train = X_tensor[train_mask]
    y_train = y_tensor[train_mask]
    meta_train = metadata[train_mask]
    
    X_test = X_tensor[test_mask]
    y_test = y_tensor[test_mask]
    meta_test = metadata[test_mask]
    
    # Create Datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    # Create Sampler for Training only
    train_sampler = P300BalancedBatchSampler(meta_train, y_train.numpy(), n_subjects_per_batch=4)
    train_loader = DataLoader(train_dataset, 
                              batch_sampler=train_sampler,
                              num_workers=3,
                              pin_memory=False)
    
    # For testing, we can use a standard DataLoader or evaluate the whole subject together
    # Choosing batch size 128 for the baseline evaluation
    test_loader = DataLoader(test_dataset, 
                             batch_size=128, 
                             shuffle=False,
                             num_workers=3,
                             pin_memory=False)
    
    # 2. MODEL INITIALIZATION (New model for each test subject)
    model = EEGInception(
        in_shape=(n_channels, n_samples),
        n_out=n_classes,
        alignment='None', 
        dropout=0.25
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 3. TRAINING LOOP
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad(set_to_none=True)  # Optimization for PyTorch 2.0+
            
            # Note on sbj_trials: In the imbalanced train_loader it is always 12
            outputs = model(batch_X, sbj_trials=12)
            
            loss = criterion(outputs, batch_y.to(device))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")
            
    # 4. EVALUATION ON TEST SUBJECT
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            # If the last batch has fewer than 12 trials, we must adjust sbj_trials
            current_batch_size = batch_X.shape[0]
            
            batch_X = batch_X.to(device)
            
            outputs = model(batch_X, sbj_trials=current_batch_size)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(batch_y.numpy())
            
    # 5. METRIC CALCULATION
    bal_acc = balanced_accuracy_score(all_targets, all_preds)
    loso_results.append(bal_acc)
    
    print(f"-> Finished Test Subject {test_subject} | Balanced Acc: {bal_acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")
    
# --- FINAL RESULTS ---
print("\n" + "="*40)
print("FINAL LOSO RESULTS (Baseline)")
print(f"Mean Balanced Accuracy: {np.mean(loso_results)*100:.2f}% ± {np.std(loso_results)*100:.2f}%")
print("="*40)

Utilizzando il device: cpu

=== Inizio Validazione LOSO (Leave-One-Subject-Out) ===

-> Addestramento per Test Subject: 1
   Epoca [5/10], Loss: 0.2190
   Epoca [10/10], Loss: 0.1773
-> Fine Test Subject 1 | Balanced Acc: 74.04% | Tempo: 1059.6s

-> Addestramento per Test Subject: 2
   Epoca [5/10], Loss: 0.3060
   Epoca [10/10], Loss: 0.2597
-> Fine Test Subject 2 | Balanced Acc: 89.49% | Tempo: 1048.4s

-> Addestramento per Test Subject: 3
   Epoca [5/10], Loss: 0.2767
   Epoca [10/10], Loss: 0.2377
-> Fine Test Subject 3 | Balanced Acc: 87.22% | Tempo: 1058.2s

-> Addestramento per Test Subject: 4
   Epoca [5/10], Loss: 0.2602
   Epoca [10/10], Loss: 0.2198
-> Fine Test Subject 4 | Balanced Acc: 57.58% | Tempo: 1042.8s

-> Addestramento per Test Subject: 5
   Epoca [5/10], Loss: 0.2124
   Epoca [10/10], Loss: 0.1765
-> Fine Test Subject 5 | Balanced Acc: 69.14% | Tempo: 1067.0s

RISULTATI FINALI LOSO (Latent Alignment)
Accuratezza Bilanciata Media: 75.49% ± 11.81%


```text
Using device: cpu

=== Starting LOSO (Leave-One-Subject-Out) Validation ===

-> Training for Test Subject: 1
   Epoch [5/10], Loss: 0.2190
   Epoch [10/10], Loss: 0.1773
-> Finished Test Subject 1 | Balanced Acc: 74.04% | Time: 1059.6s

-> Training for Test Subject: 2
   Epoch [5/10], Loss: 0.3060
   Epoch [10/10], Loss: 0.2597
-> Finished Test Subject 2 | Balanced Acc: 89.49% | Time: 1048.4s

-> Training for Test Subject: 3
   Epoch [5/10], Loss: 0.2767
   Epoch [10/10], Loss: 0.2377
-> Finished Test Subject 3 | Balanced Acc: 87.22% | Time: 1058.2s

-> Training for Test Subject: 4
   Epoch [5/10], Loss: 0.2602
   Epoch [10/10], Loss: 0.2198
-> Finished Test Subject 4 | Balanced Acc: 57.58% | Time: 1042.8s

-> Training for Test Subject: 5
   Epoch [5/10], Loss: 0.2124
   Epoch [10/10], Loss: 0.1765
-> Finished Test Subject 5 | Balanced Acc: 69.14% | Time: 1067.0s

============================================================
FINAL LOSO RESULTS (Latent Alignment)
Mean Balanced Accuracy: 75.49% ± 11.81%
============================================================